# 03 — Aprendizaje supervisado: reconocimiento de actividades humanas


Entrenar y comparar dos clasificadores multiclase para reconocer seis actividades humanas a partir de 561 variables de sensores de teléfonos inteligentes. Se parte de los archivos preparados en `02_preprocesamiento.ipynb` y se mantiene separada la prueba oficial del dataset HAR (personas no vistas durante el entrenamiento).

**Modelos:** regresión logística y Random Forest. **Métricas:** Accuracy, Macro F1-score, informe por clase y matriz de confusión.

> **Instrucción:** ejecutar primero `02_preprocesamiento.ipynb` para crear `data/procesado/train.pkl` y `test.pkl`. Este notebook no modifica ni desarrolla `04_no_supervisado`.

## 1. Bibliotecas y carga de datos
Los archivos `.pkl` tienen 561 variables predictoras y cuatro columnas auxiliares: `actividad_id`, `actividad`, `grupo` y `persona`. La columna `grupo` representa estática/dinámica, por lo que **no debe entrar al clasificador**, al igual que las otras columnas auxiliares.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold, cross_validate
from sklearn.metrics import (accuracy_score, f1_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay)

SEED = 42
np.random.seed(SEED)

# Compatible con ejecución desde la carpeta notebooks/ o desde la raíz evaluacion 4/.
carpetas = [Path('../data/procesado'), Path('data/procesado'),
            Path('evaluacion 4/data/procesado')]
data_dir = next((d for d in carpetas if (d / 'train.pkl').exists() and
                 (d / 'test.pkl').exists()), None)
if data_dir is None:
    raise FileNotFoundError(
        'No se encontraron train.pkl y test.pkl. Ejecuta primero 02_preprocesamiento.ipynb '
        'con el dataset HAR descargado en evaluacion 4/data/. '
        'Si trabajas en Colab, sube estos archivos junto al notebook y ajusta data_dir.')

train = pd.read_pickle(data_dir / 'train.pkl')
test = pd.read_pickle(data_dir / 'test.pkl')
auxiliares = ['actividad_id', 'actividad', 'grupo', 'persona']
assert all(c in train.columns and c in test.columns for c in auxiliares)
X_train = train.drop(columns=auxiliares)
X_test = test.drop(columns=auxiliares)
y_train = train['actividad_id'].astype(int)
y_test = test['actividad_id'].astype(int)
s_train = train['persona']
assert list(X_train.columns) == list(X_test.columns), 'Las columnas de entrenamiento y prueba deben coincidir'
assert not X_train.isna().any().any() and not X_test.isna().any().any()
assert set(s_train).isdisjoint(set(test['persona'])), 'La prueba debe contener participantes diferentes'

etiquetas = sorted(set(y_train) | set(y_test))
nombres = (pd.concat([train[['actividad_id','actividad']],test[['actividad_id','actividad']]])
           .drop_duplicates('actividad_id').set_index('actividad_id')['actividad'].to_dict())
nombres_ordenados = [nombres[i] for i in etiquetas]
print('Entrenamiento:', X_train.shape, '| Prueba:', X_test.shape)
print('Número de actividades:', len(etiquetas))
print('Participantes entrenamiento:', s_train.nunique(), '| prueba:', test['persona'].nunique())
print('Clases:', nombres)

## 2. Modelos y justificación

**Regresión logística multinomial:** sirve como línea base interpretable y eficiente para clasificación de varias clases. Requiere escalar las variables porque utiliza regularización; el `StandardScaler` se ajusta exclusivamente sobre los datos de cada partición de entrenamiento mediante `Pipeline`. Sus fronteras de decisión son lineales, lo que puede limitarla ante relaciones complejas.

**Random Forest:** combina árboles de decisión y captura relaciones no lineales e interacciones entre variables. No requiere escalamiento, pero puede ser más costoso computacionalmente y menos interpretable. Su importancia de variables se debe tratar con cautela ante características altamente correlacionadas.

Se utiliza `GroupKFold` por `persona` para que los sujetos de entrenamiento y validación sean distintos. La prueba oficial se reserva para evaluación final.

In [ ]:
modelos = {
    'Regresión logística': Pipeline([
        ('escalado', StandardScaler()),
        ('modelo', LogisticRegression(C=0.1, max_iter=3000, random_state=SEED))
    ]),
    'Random Forest': RandomForestClassifier(
        n_estimators=200, min_samples_leaf=2,
        random_state=SEED, n_jobs=-1
    )
}
validacion = GroupKFold(n_splits=5)
filas_cv = []
for nombre, modelo in modelos.items():
    puntajes = cross_validate(
        modelo, X_train, y_train, groups=s_train, cv=validacion,
        scoring={'accuracy':'accuracy','macro_f1':'f1_macro'}, n_jobs=1
    )
    filas_cv.append({
        'Modelo': nombre,
        'Accuracy CV (media)': puntajes['test_accuracy'].mean(),
        'Accuracy CV (desv.)': puntajes['test_accuracy'].std(),
        'Macro F1 CV (media)': puntajes['test_macro_f1'].mean(),
        'Macro F1 CV (desv.)': puntajes['test_macro_f1'].std(),
    })
comparacion_cv = pd.DataFrame(filas_cv).sort_values('Macro F1 CV (media)', ascending=False)
display(comparacion_cv.round(4))

## 3. Entrenamiento definitivo y evaluación con participantes no vistos

Se ajusta cada algoritmo con **todo el conjunto de entrenamiento** y se evalúa una sola vez con `test.pkl`. No se usan las observaciones de prueba para ajustar parámetros ni transformaciones.

In [ ]:
modelos_ajustados = {}
predicciones = {}
filas_test = []
for nombre, modelo in modelos.items():
    ajustado = clone(modelo).fit(X_train, y_train)
    pred = ajustado.predict(X_test)
    modelos_ajustados[nombre] = ajustado
    predicciones[nombre] = pred
    filas_test.append({
        'Modelo': nombre,
        'Accuracy test': accuracy_score(y_test, pred),
        'Macro F1 test': f1_score(y_test, pred, average='macro'),
    })
comparacion_test = pd.DataFrame(filas_test).sort_values('Macro F1 test', ascending=False)
display(comparacion_test.round(4))
for nombre, pred in predicciones.items():
    print(f'\n=== {nombre} — reporte por actividad ===')
    print(classification_report(y_test, pred, labels=etiquetas,
                                target_names=nombres_ordenados, zero_division=0))

## 4. Visualizaciones: comparación y matrices de confusión

Las matrices permiten estudiar en qué actividades aparecen errores, especialmente entre movimientos parecidos como caminar y subir o bajar escaleras. Las filas representan la actividad verdadera y las columnas la predicha.

In [ ]:
ax = comparacion_test.set_index('Modelo')[['Accuracy test','Macro F1 test']].plot(
    kind='bar', figsize=(8,4), rot=0, ylim=(0,1), title='Desempeño de los modelos en prueba')
ax.set_ylabel('Puntuación')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

for nombre, pred in predicciones.items():
    fig, ax = plt.subplots(figsize=(9,7))
    ConfusionMatrixDisplay.from_predictions(
        y_test, pred, labels=etiquetas, display_labels=nombres_ordenados,
        normalize='true', cmap='Blues', values_format='.2f', ax=ax,
        xticks_rotation=40, colorbar=False
    )
    ax.set_title(f'Matriz de confusión normalizada — {nombre}')
    plt.tight_layout()
    plt.show()

## 5. Actividades con mayor confusión

A continuación se presentan los errores más frecuentes de cada modelo, excluyendo los aciertos de la diagonal. Su interpretación debe relacionarse con similitudes físicas entre las actividades.

In [ ]:
for nombre, pred in predicciones.items():
    matriz = confusion_matrix(y_test, pred, labels=etiquetas)
    np.fill_diagonal(matriz, 0)
    confusiones = []
    for i, j in zip(*np.where(matriz > 0)):
        confusiones.append((nombres_ordenados[i], nombres_ordenados[j], int(matriz[i,j])))
    confusiones.sort(key=lambda r: r[2], reverse=True)
    print(f'\n{nombre}: principales confusiones (real → predicha)')
    display(pd.DataFrame(confusiones[:8], columns=['Actividad real','Predicha como','Casos']))

## 6. Respuesta a la pregunta del equipo: ¿estático vs. dinámico es más fácil?

La etiqueta binaria estática/dinámica se **deriva de las predicciones de seis clases ya obtenidas**. Por tanto, se trata de una comparación descriptiva para entender los errores, **no de un tercer clasificador entrenado**. Esta comparación puede dar una puntuación mayor porque los errores dentro de un mismo grupo dejan de contar como errores.

In [ ]:
dinamicas = {1,2,3}  # WALKING, WALKING_UPSTAIRS y WALKING_DOWNSTAIRS
real_bin = np.array(['dinámica' if int(v) in dinamicas else 'estática' for v in y_test])
comparacion_grupos = []
for nombre, pred in predicciones.items():
    pred_bin = np.array(['dinámica' if int(v) in dinamicas else 'estática' for v in pred])
    comparacion_grupos.append({
        'Modelo': nombre,
        'Accuracy 6 actividades': accuracy_score(y_test,pred),
        'Accuracy estática/dinámica (derivada)':accuracy_score(real_bin,pred_bin)
    })
display(pd.DataFrame(comparacion_grupos).round(4))

## 7. Exportación de métricas y predicciones

Los archivos CSV se guardan en `resultados_supervisado/` junto a los datos procesados. Se pueden subir a GitHub y usar como visualizaciones complementarias, o importarlos en Power BI **solo si el grupo lo desea**.

In [ ]:
carpeta_salida = data_dir.parent / 'resultados_supervisado'
carpeta_salida.mkdir(parents=True, exist_ok=True)
comparacion_cv.to_csv(carpeta_salida / 'metricas_validacion.csv', index=False)
comparacion_test.to_csv(carpeta_salida / 'metricas_prueba.csv', index=False)
salida = pd.DataFrame({'actividad_real_id': y_test.to_numpy(),
                       'actividad_real': [nombres[int(y)] for y in y_test]})
for nombre, pred in predicciones.items():
    sufijo = 'logistica' if nombre == 'Regresión logística' else 'random_forest'
    salida[f'pred_{sufijo}_id'] = pred
    salida[f'pred_{sufijo}'] = [nombres[int(y)] for y in pred]
salida.to_csv(carpeta_salida / 'predicciones_test.csv', index=False)
print('Archivos generados en:', carpeta_salida)

## 8. Interpretación y conclusiones (completar después de ejecutar)

**No escribir resultados numéricos antes de ejecutar el código.** Con base en las tablas y gráficos anteriores, completar:

1. **Desempeño comparado:** indicar cuál modelo tuvo mayor Macro F1-score y Accuracy en prueba, con cifras reales.
2. **Errores entre actividades:** identificar el par de actividades que más se confundió y explicar una causa plausible según las características de su movimiento.
3. **Generalización:** valorar que entrenamiento y prueba contienen personas diferentes; señalar que nuevos dispositivos, ubicaciones del teléfono y poblaciones distintas podrían producir resultados diferentes.
4. **Pregunta principal:** contrastar la clasificación de seis actividades con la distinción estática/dinámica derivada de las predicciones.
5. **Elección para aplicación práctica:** recomendar uno de los modelos con una justificación que considere desempeño, tiempo computacional e interpretabilidad.

### Limitaciones
Las 561 características se calcularon sobre ventanas de sensores del conjunto HAR UCI; no se ha probado el sistema en tiempo real ni con dispositivos distintos. La validación por persona reduce el riesgo de resultados demasiado optimistas, pero no garantiza generalización a otras condiciones.

### Articulación con el componente 04
Entregar a la persona encargada de `04_no_supervisado` los archivos y métricas resultantes para apoyar la discusión final. Los modelos no supervisados deben entrenarse **sin usar la etiqueta de actividad como variable de entrada**.